# MicroDuck 键盘遥操作：策略推理验证

这个实验加载已经训练好的 `model_*.pt`，只创建 **1 台 MicroDuck**，在 MuJoCo 中以 **50 Hz** 执行策略推理。键盘不直接控制关节，而是修改期望速度 `(vx, vy, yaw_rate)`；PPO Actor 根据实时观测生成 14 维关节动作。

```text
键盘速度命令 → 61D 观测 → PPO Actor → 14D 关节目标 → BAM 执行器 → MuJoCo
      ↑                                                         │
      └────────────── 新姿态、速度和传感器观测 ──────────────────┘
```

这验证的是：checkpoint 能加载、观测维度匹配、Actor 能实时推理、动作能驱动机器人，以及策略对人工速度命令有闭环响应。

> 这仍是**仿真推理验证**，不等于完成真机部署。真机还需要模型导出/数值一致性、硬件接口、控制周期、延迟、动作限幅、跌倒保护和急停验证。

## 使用前提

1. 使用更新后的镜像启动 lab（包含浏览器 MuJoCo 桌面）；
2. 本实验先用内置、自动筛选的 950-iter Reference Demo 验证 teleop 链路，建立“正常响应”的基准；
3. 再加载 `01_velocity_lab.ipynb` 训练的 300-iter 模型，在相同命令下比较稳定性和跟踪效果。

## 1. 先用稳定 Reference Demo 验证 teleop 链路

下面加载 `examples/velocity_flat_demo/model_950.pt`，创建一个 MuJoCo 环境和一台机器人。该模型不是课堂训练产物，而是经过定向微调，并从多个 checkpoint 中用六方向固定命令自动评测选出的参考资产：

- 每个方向使用 32 个并行 trial，持续 500 steps（10 秒）；
- 前、后、左、右四个平移方向全部通过稳定性、方向和误差门槛；
- 在 `yaw_rate=±0.8 rad/s` 时，seed-123 的左右转平均方向正确且 10 秒稳定率为 100%；跨 seed 右转仍弱于平移，因此它是改进后的参考而不是完美控制器。

先确认：

1. 嵌入画面和键盘焦点正常；
2. 方向键与 Q/E 命令能到达策略；
3. 机器人能对前后、横移和转向命令产生可观察响应；
4. 绿色目标线速度箭头与蓝色实际线速度箭头能够显示。

确认 Reference Demo 链路正常后，再到后面的步骤切换为学员模型进行对比。完整量化结果保存在 `examples/velocity_flat_demo/cardinal_eval.json`。

In [ ]:
# 稳定基准：直接启动自动筛选的 950-iter Reference Demo。
!bash /workspace/microduck_rl_tutorial/scripts/run_teleop.sh demo

In [ ]:
# 确认 teleop 进程真实存在后，再将 noVNC 实时桌面嵌入当前 Notebook。
# iframe 主机名 = 浏览器地址栏里的 Jupyter 主机（含跳板）；端口 = 该主机上的 noVNC。
import sys

sys.path.insert(0, "/workspace/microduck_rl_tutorial/scripts")
from microduck_novnc_embed import display_teleop_frame

display_teleop_frame(lang="zh")

## 2. 键盘控制（绝对方向模式）

连接后可双击嵌入桌面中的 MuJoCo 窗口标题栏将其最大化，再点击机器人画面，使键盘焦点进入嵌入窗口。四个方向键控制平移，`Q/E` 控制转向；WASD 不再控制机器人。如果方向键滚动了 Notebook，说明焦点尚未进入桌面，请再点击一次机器人画面。

| 按键 | 设置的完整命令 `(vx, vy, yaw_rate)` |
|---|---|
| `↑` | `(+0.2, 0, 0)`：前进 |
| `↓` | `(-0.2, 0, 0)`：后退 |
| `←` | `(0, +0.2, 0)`：向左横移 |
| `→` | `(0, -0.2, 0)`：向右横移 |
| `Q` | `(0, 0, +0.8)`：向左转 |
| `E` | `(0, 0, -0.8)`：向右转 |
| `X` 或 `Enter` | `(0, 0, 0)`：停止 |
| `Backspace` | reset 机器人并清零命令 |
| `Space` | 恢复/继续；为防止误操作，不会进入暂停 |
| `N` | 主动进入暂停状态并单步执行 |
| `+ / -` | 调整播放速度 |

每个运动键都会设置固定速度并清除其他轴；重复按键不会继续加速。此模式分别验证前后、横移和原地转向，暂不支持组合命令。

### 如何观察速度箭头与机体坐标系

- 所有命令都使用**机器人机体坐标系**，不是屏幕或世界坐标：`+vx` 是机器人面朝方向，`+vy` 是机器人自身左侧，`+yaw` 是从上方看逆时针左转；拖动相机后，屏幕上的“左/右”可能与机体左/右不同；
- 按 `↑/↓/←/→` 后，机器人头顶会出现两个箭头：**绿色**表示键盘设定的目标线速度，**蓝色**表示从 MicroDuck `EntityData` 读取的实际机体线速度；
- 两个箭头的起点被故意横向错开约 `0.1 m`，用于避免颜色相互遮挡。因此，**不要用两个箭头之间的空间距离判断误差**；应比较它们的方向和长度；
- 蓝色箭头显示瞬时速度，会随着左右脚交替和身体摆动而抖动。应观察几秒内它是否总体跟随绿色箭头，而不是要求每一帧完全重合；
- 箭头方向表示运动方向，长度表示速度大小。两者方向一致、长度接近，说明跟踪较好；蓝色长期反向或明显更短，才表示方向或速度跟踪不足；
- 命令归零时绿色箭头会消失；相机持续跟随机器人位置，但保留用户选择的观察角度；
- `Q/E` 设置的是 yaw 角速度，当前 viewer 只绘制线速度箭头，因此原地转向时没有绿色箭头，应观察机器人旋转方向和日志中的 `vyaw=±0.80`；
- 若箭头被机器人遮挡，可在 MuJoCo 画面中拖动视角或滚轮缩放。

> 若日志中出现 `paused` 且机器人不动，在机器人画面内按一次 `Space` 恢复。浏览器 teleop 已禁用 Space 的暂停切换，以避免 Notebook 滚动操作意外冻结仿真。

## 3. 切换到学员的 300-iter 模型做对比

先记录 Demo 在前后、横移和转向命令下的表现。然后将下一格的 `LOAD_STUDENT_MODEL` 改为 `True` 并单独运行；脚本会寻找最近一次完成 300 iter 的 run，停止 Demo viewer，再加载对应的 `model_*.pt`。

使用相同按键进行对比，重点观察：响应方向、episode 持续时间、漂移、抖动和摔倒频率。默认值为 `False`，因此 **Run All 不会意外切换模型**。

In [ ]:
# 为避免 Run All 意外替换 Reference Demo，默认不执行。
LOAD_STUDENT_MODEL = False

if LOAD_STUDENT_MODEL:
    !bash /workspace/microduck_rl_tutorial/scripts/run_teleop.sh latest
    print("学员模型已启动。请回到嵌入窗口点击‘重新连接’，再使用相同命令对比。")
else:
    print("继续使用 Reference Demo。需要对比时，将 LOAD_STUDENT_MODEL 改为 True 后单独运行本 cell。")

### 如何解释 Reference Demo 与学员模型的差异

这一步不仅比较“哪个走得更好”，还用于定位问题发生在哪一层：

- **Reference Demo 能稳定响应**：说明 checkpoint 加载、61 维观测、PPO Actor 推理、键盘命令注入、MuJoCo 和画面传输链路都正常；
- **切换学员模型后不明显移动、原地踏步、漂移或跌倒**：结合 `01` 中该模型较低的 reward 和较短的 episode length，应判断为策略尚未形成可靠步态，而不是 teleop 或网络失效；
- **绿色目标箭头随方向键立即变化，但蓝色箭头和机器人响应弱**：命令已经送达，差异主要来自策略和机器人动力学响应；
- **绿色箭头也不变化，且日志没有新命令**：再检查嵌入窗口焦点和键盘传输。

Reference Demo 的改进不只是“训练更久”：自动 checkpoint sweep 发现旧 500-iter 模型横移幅值很低且右转平均方向错误；之后通过均衡六方向采样、强化速度 tracking reward、提高 yaw 样本和左右镜像约束，并按固定命令评测选择 `model_950`。这说明训练 reward 上升和 final checkpoint 都不能替代面向目标行为的验收。

本次课堂生成的 300-iter checkpoint 平均 episode 仅约 1–2 秒，因此它更适合作为“训练不足的反例”，不应被当作稳定部署模型。先用 Reference Demo 建立正常基准，再比较学员模型，可以避免把模型质量问题误判为推理部署故障。

> 这一判断只针对本次实际 checkpoint。其他硬件、随机种子或训练配置产生的 300-iter 模型可能有不同表现，必须重新结合指标与视频判断。

## 4. 停止交互推理

关闭、滚离嵌入窗口或离开 Notebook 只会断开画面，不会停止后台推理。完成实验后，将下一格的 `STOP_TELEOP` 改为 `True` 并单独运行，以释放 GPU 和仿真资源。默认值为 `False`，因此 **Run All 不会导致嵌入窗口黑屏**。

In [ ]:
# 安全开关：Run All 时不会自动关闭 viewer。
STOP_TELEOP = False

if STOP_TELEOP:
    !bash /workspace/microduck_rl_tutorial/scripts/run_teleop.sh stop
else:
    print("Teleop 保持运行。完成实验后，将 STOP_TELEOP 改为 True 并单独运行本 cell。")

## 5. 推理验证清单

| 检查项 | 通过标准 |
|---|---|
| Checkpoint 加载 | 先显示 Reference Demo `model_950.pt`，切换后显示学员 `model_*.pt`，均无维度错误 |
| 推理实时性 | viewer 接近实时运行，操作后持续更新，无明显长时间卡顿 |
| 目标线速度 | 方向键按下后绿色箭头立即改变；蓝色箭头来自实际机体速度；Q/E 不显示线速度箭头 |
| 坐标语义 | 前后左右按机器人自身朝向判断，不按屏幕方向判断 |
| 静止命令 | 归零后机器人能够停止或保持相对稳定，而不是持续漂移 |
| 前后响应 | `vx` 正负变化时运动方向随之变化 |
| 横向响应 | `vy` 正负变化时能向左右侧移 |
| 转向响应 | `yaw_rate` 正负变化时能向对应方向旋转；允许 yaw 幅值跟踪弱于平移 |
| 单轴绝对命令 | 切换运动键时，上一轴命令被清零，不出现意外叠加 |
| 模型差异 | 学员模型可能比 Reference Demo 更易漂移、抖动或提前摔倒，且 episode 更短 |
| 抗扰与恢复 | 命令突变后不过度振荡；reset 后能重新开始 |

如果 checkpoint 能加载但机器人不响应键盘，先确认嵌入桌面和 MuJoCo 窗口获得焦点，并观察终端日志中的 `(vx, vy, vyaw)` 是否变化。日志路径为 `/workspace/runs/logs/teleop.log`。

下一步：打开 [`03_quiz.ipynb`](03_quiz.ipynb) 完成本课 5 道选择题。
